# 20260923 Trial Prediction Review

Use this companion notebook after `20260923_trial_classification.ipynb` has saved a model to `ML_model/` and `scripts/predict_trial_labels.py` has written saved-model predictions.

This notebook is meant for student review: it shows each segmented trial trajectory with the predicted label, then saves accepted or corrected labels back into the pipeline label table.

## Order Of Operations

1. Train/save a model in `20260923_trial_classification.ipynb`.
2. Run `scripts/predict_trial_labels.py` to predict labels for processed trials.
3. Open this notebook to accept or correct those predictions from trial plots.

## Imports

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import joblib
except ImportError:
    joblib = None

from preprocess_functions.trial_classification import (
    DEFAULT_LABEL_OPTIONS,
    load_labels,
    load_trial_catalog,
    make_aligned_loader,
    plot_trial,
    upsert_label,
)

## Config

In [ ]:
OUTPUT_ROOT = Path("preprocess_out")
CLASSIFICATION_DIR = OUTPUT_ROOT / "trial_classification"
ML_MODEL_DIR = Path("ML_model")
MODEL_PATH = ML_MODEL_DIR / "trial_type_classifier.joblib"

PREDICTIONS_CSV = CLASSIFICATION_DIR / "saved_model_trial_predictions.csv"
LABELS_CSV = CLASSIFICATION_DIR / "trial_labels.csv"
REVIEW_LOG_CSV = CLASSIFICATION_DIR / "saved_model_review_log.csv"

SHOW_VIDEO_BACKGROUND = False
INVERT_Y_AXIS = True

print("model:", MODEL_PATH)
print("predictions:", PREDICTIONS_CSV)
print("labels:", LABELS_CSV)

## Run Prediction Script If Needed

In [ ]:
print(
    "python scripts/predict_trial_labels.py "
    f"--output-root {OUTPUT_ROOT} --model {MODEL_PATH} --predictions-out {PREDICTIONS_CSV}"
)

if not MODEL_PATH.exists():
    print("Missing saved model. Train/save it in 20260923_trial_classification.ipynb first.")
if not PREDICTIONS_CSV.exists():
    print("Missing predictions. Run the command above before using the review GUI.")

## Load Predictions And Trial Catalog

In [ ]:
if not PREDICTIONS_CSV.exists():
    raise FileNotFoundError(f"Run prediction first: {PREDICTIONS_CSV}")

pred_df = pd.read_csv(PREDICTIONS_CSV)
trial_catalog, aligned_index, trial_index = load_trial_catalog(OUTPUT_ROOT)
load_aligned_for_row = make_aligned_loader()

if joblib is not None and MODEL_PATH.exists():
    model_bundle = joblib.load(MODEL_PATH)
    label_options = model_bundle.get("label_options", DEFAULT_LABEL_OPTIONS)
else:
    model_bundle = {}
    label_options = DEFAULT_LABEL_OPTIONS

pred_df["trial_idx"] = pd.to_numeric(pred_df["trial_idx"], errors="coerce").astype("Int64")
print(f"Loaded {len(pred_df)} predictions from {pred_df['recording_id'].nunique()} recordings")
display(pred_df.head())

## Review GUI

In [ ]:
def catalog_row_for_prediction(pred_row: pd.Series) -> pd.Series:
    match = trial_catalog[
        (trial_catalog["recording_id"].astype(str) == str(pred_row["recording_id"]))
        & (pd.to_numeric(trial_catalog["trial_idx"], errors="coerce") == int(pred_row["trial_idx"]))
    ]
    if match.empty:
        raise KeyError("Prediction row is not present in trial_catalog")
    return match.iloc[0]


def append_review_log(pred_row: pd.Series, chosen_label: str, action: str, notes: str = "") -> None:
    entry = pd.DataFrame([
        {
            "recording_id": pred_row.get("recording_id"),
            "session_id": pred_row.get("session_id"),
            "trial_idx": int(pred_row.get("trial_idx")),
            "pred_label": pred_row.get("pred_label"),
            "pred_conf": pred_row.get("pred_conf"),
            "chosen_label": chosen_label,
            "review_action": action,
            "notes": notes,
            "model_name": pred_row.get("model_name"),
            "model_path": pred_row.get("model_path"),
        }
    ])
    REVIEW_LOG_CSV.parent.mkdir(parents=True, exist_ok=True)
    if REVIEW_LOG_CSV.exists():
        log = pd.read_csv(REVIEW_LOG_CSV)
        log = pd.concat([log, entry], ignore_index=True)
    else:
        log = entry
    log.to_csv(REVIEW_LOG_CSV, index=False)


recordings = sorted(pred_df["recording_id"].dropna().astype(str).unique())
pred_labels = ["(all)"] + sorted(pred_df["pred_label"].dropna().astype(str).unique())

recording_dropdown = widgets.Dropdown(options=recordings, description="Recording:", layout=widgets.Layout(width="420px"))
pred_filter_dropdown = widgets.Dropdown(options=pred_labels, value="(all)", description="Pred:")
low_conf_first = widgets.Checkbox(value=True, description="low conf first")
unreviewed_only = widgets.Checkbox(value=True, description="unreviewed only")
trial_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Trial:", continuous_update=False)
correction_dropdown = widgets.Dropdown(options=label_options, value=label_options[0], description="Correct:", layout=widgets.Layout(width="320px"))
notes_text = widgets.Text(value="", description="Notes:", placeholder="optional", layout=widgets.Layout(width="420px"))
accept_button = widgets.Button(description="Accept pred", button_style="success")
correct_button = widgets.Button(description="Save correction", button_style="warning")
skip_button = widgets.Button(description="Skip")
back_button = widgets.Button(description="Back")
reload_button = widgets.Button(description="Reload")
status_out = widgets.Output()
plot_out = widgets.Output()

visible_indices: list[int] = []


def reviewed_keys() -> set[tuple[str, int]]:
    labels = load_labels(LABELS_CSV)
    if labels.empty:
        return set()
    return set(zip(labels["recording_id"].astype(str), pd.to_numeric(labels["trial_idx"], errors="coerce").fillna(-1).astype(int)))


def compute_visible_indices() -> list[int]:
    subset = pred_df[pred_df["recording_id"].astype(str) == str(recording_dropdown.value)].copy()
    if pred_filter_dropdown.value != "(all)":
        subset = subset[subset["pred_label"].astype(str) == str(pred_filter_dropdown.value)]
    if unreviewed_only.value:
        keys = reviewed_keys()
        subset = subset[
            ~subset.apply(lambda r: (str(r["recording_id"]), int(r["trial_idx"])) in keys, axis=1)
        ]
    if low_conf_first.value and "pred_conf" in subset.columns:
        subset = subset.sort_values("pred_conf", ascending=True, na_position="first")
    return subset.index.tolist()


def current_prediction_row() -> pd.Series | None:
    if not visible_indices:
        return None
    return pred_df.loc[visible_indices[min(trial_slider.value, len(visible_indices) - 1)]]


def refresh_options(*_):
    global visible_indices
    visible_indices = compute_visible_indices()
    trial_slider.max = max(0, len(visible_indices) - 1)
    trial_slider.value = min(trial_slider.value, trial_slider.max)
    refresh_plot()


def refresh_plot(*_):
    row = current_prediction_row()
    with plot_out:
        clear_output(wait=True)
        if row is None:
            print("No predictions for this filter.")
            return
        catalog_row = catalog_row_for_prediction(row)
        manual = row.get("manual_label", "")
        pred_text = f"pred={row.get('pred_label')} conf={row.get('pred_conf', np.nan):.2f} manual={manual}"
        fig = plot_trial(
            catalog_row,
            load_aligned_for_row=load_aligned_for_row,
            title_prefix="Saved-model prediction review",
            prediction_text=pred_text,
            show_video_background=SHOW_VIDEO_BACKGROUND,
            invert_y_axis=INVERT_Y_AXIS,
        )
        plt.show(fig)
        plt.close(fig)
    with status_out:
        clear_output(wait=True)
        print(f"Visible {trial_slider.value + 1}/{len(visible_indices)} | predictions: {PREDICTIONS_CSV}")
        print(f"labels: {LABELS_CSV}")


def advance():
    if trial_slider.value < trial_slider.max:
        trial_slider.value += 1
    else:
        refresh_options()


def save_choice(label: str, action: str):
    row = current_prediction_row()
    if row is None:
        return
    catalog_row = catalog_row_for_prediction(row)
    upsert_label(
        load_labels(LABELS_CSV),
        catalog_row,
        label,
        path=LABELS_CSV,
        notes=notes_text.value,
        extra={
            "label_source": "saved_model_review",
            "pred_label": row.get("pred_label"),
            "pred_conf": row.get("pred_conf"),
            "model_name": row.get("model_name"),
        },
    )
    append_review_log(row, label, action, notes_text.value)
    notes_text.value = ""
    advance()


def accept_prediction(_=None):
    row = current_prediction_row()
    if row is not None:
        save_choice(str(row["pred_label"]), "accepted_prediction")


def save_correction(_=None):
    save_choice(str(correction_dropdown.value), "corrected_prediction")


def skip_current(_=None):
    advance()


def back_one(_=None):
    if trial_slider.value > 0:
        trial_slider.value -= 1
    else:
        refresh_plot()


recording_dropdown.observe(refresh_options, names="value")
pred_filter_dropdown.observe(refresh_options, names="value")
low_conf_first.observe(refresh_options, names="value")
unreviewed_only.observe(refresh_options, names="value")
trial_slider.observe(refresh_plot, names="value")
accept_button.on_click(accept_prediction)
correct_button.on_click(save_correction)
skip_button.on_click(skip_current)
back_button.on_click(back_one)
reload_button.on_click(refresh_options)

display(widgets.VBox([
    widgets.HBox([recording_dropdown, pred_filter_dropdown]),
    widgets.HBox([low_conf_first, unreviewed_only]),
    trial_slider,
    widgets.HBox([correction_dropdown, notes_text]),
    widgets.HBox([accept_button, correct_button, skip_button, back_button, reload_button]),
    status_out,
    plot_out,
]))
refresh_options()

## Output Files

- `preprocess_out/trial_classification/saved_model_trial_predictions.csv`: predictions created by the script.
- `preprocess_out/trial_classification/trial_labels.csv`: accepted/corrected labels used by future training runs.
- `preprocess_out/trial_classification/saved_model_review_log.csv`: append-only record of review actions.